<a href="https://colab.research.google.com/github/panavgohil/ML-Assignment-1/blob/main/House_Price_Prediction_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVR, SVC
from sklearn.linear_model import SGDRegressor, SGDClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)


In [ ]:
!pip install imbalanced-learn  #SMOTE Library

In [ ]:
from imblearn.over_sampling import SMOTE    #SMOTE creates artificial samples of the minority class, model learns fairly

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving kc_house_data.csv to kc_house_data.csv


In [ ]:
#load Dataset

df = pd.read_csv("kc_house_data.csv")

print("Rows:", len(df))
print("Columns:", df.shape[1])

df.head()


Rows: 21613
Columns: 21


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long  

In [ ]:
#data preprocessing -->cleanup extra columns
# Remove unnecessary columns

for col in ["id", "date"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True) #inplace=True -->modify in same dataframe

df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [ ]:
df.isnull().sum()

,0
price,0
bedrooms,0
bathrooms,0
sqft_living,0
sqft_lot,0
floors,0
waterfront,0
view,0
condition,0
grade,0


In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
TARGET = "price"

FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES].copy()
y_reg = df[TARGET].copy()

In [ ]:
print(X.shape)
print(y_reg.shape)

(21613, 18)
(21613,)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train, y_test=train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)
print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

Training data: (17290, 18)
Testing data : (4323, 18)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
print(X_train.iloc[0])
print("\nScaled:\n")
print(X_train_scaled[0])

bedrooms             3.000
bathrooms            1.750
sqft_living       1780.000
sqft_lot         13095.000
floors               1.000
waterfront           0.000
view                 0.000
condition            4.000
grade                9.000
sqft_above        1780.000
sqft_basement        0.000
yr_built          1983.000
yr_renovated         0.000
zipcode          98042.000
lat                 47.367
long              -122.152
sqft_living15     2750.000
sqft_lot15       13095.000
Name: 6325, dtype: float64

Scaled:

[-0.39526335 -0.47445144 -0.32393262 -0.04387306 -0.91959976 -0.08499166
 -0.30591651  0.90907268  1.15024328 -0.00725676 -0.65631017  0.40400107
 -0.20829394 -0.6746308  -1.39660754  0.44228847  1.12607326  0.01344043]


In [ ]:
#dividing house prices into low, mid, and high categories
q33=y_reg.quantile(.33)
q66=y_reg.quantile(.66)
print("33rd percentile",q33)
print("66th percentile",q66)

33rd percentile 360000.0
66th percentile 560000.0


In [ ]:
y_cls=pd.cut(
    y_reg,
    bins=[-np.inf, q33,q66,np.inf],
    labels=["low","mid","high"]
)
print(y_cls.head(10))

0     low
1     mid
2     low
3    high
4     mid
5    high
6     low
7     low
8     low
9     low
Name: price, dtype: category
Categories (3, object): ['low' < 'mid' < 'high']


In [18]:
print(y_cls.value_counts())

price
high    7296
low     7226
mid     7091
Name: count, dtype: int64


In [20]:
mask_high=y_cls=='high'
drop_idx=y_cls[mask_high].sample(
    frac=0.55,
    random_state=42
).index
df_bal=df.drop(index=drop_idx).reset_index(drop=True)
print("Original rows:", len(df))
print("New rows:", len(df_bal))

Original rows: 21613
New rows: 17600


In [21]:
X_cls=df_bal[FEATURES].copy()
y_cls2=pd.cut(
    df_bal["price"],
    bins=[-np.inf,q33,q66,np.inf],
    labels=["low","mid","high"]
)

In [23]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_cls_enc=le.fit_transform(y_cls2)
print("classes:",le.classes_)
print("First 10 labels:",y_cls_enc[:10])

classes: ['high' 'low' 'mid']
First 10 labels: [1 2 1 2 0 1 1 1 1 0]


In [24]:
import numpy as np

print(
    dict(
        zip(
            le.classes_,
            np.bincount(y_cls_enc)
        )
    )
)

{'high': np.int64(3283), 'low': np.int64(7226), 'mid': np.int64(7091)}


# House Sales Prediction and Classification using SVM

## Name: Panav Gohil
## Roll number: 25/A14/031

### Objective
To predict house prices using Support Vector Regression (SVR) and classify houses into price categories using Support Vector Classification (SVC). The project also demonstrates handling imbalanced data using SMOTE.